# Stage 3 — Normalization / Encoding + Patient-Level Train/Test Split
**Capstone: Multimodal Ocular Disease Prediction (Myopia focus — Experiment 1)**

Pipeline position: Data Exploration → Data Cleaning + Feature ID → **Normalization / Encoding + Patient-level Split** → Classification → Evaluation

This notebook does NOT train any model. It:
1. Loads `Data/myopia_baseline_clean.csv` (Stage 2 output).
2. Investigates per-patient label consistency (needed to design the split correctly).
3. Builds a patient-level, stratified train/test split using `ID` — no patient appears in both sets.
4. Fits imputation and scaling **on the training data only**, then applies it to both sets.
5. Encodes `Patient Sex`.
6. Exports `Data/train_set.csv` and `Data/test_set.csv`, and shows the final feature matrix.

Project folder layout:
```
Capstone Datasets/
├── Data/            <- myopia_baseline_clean.csv lives here; train_set.csv / test_set.csv saved here
├── Code/
│   └── notebooks/   <- this notebook lives here
└── Documentation/
```

## 1. Load the cleaned dataset (Stage 2 output)

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option("display.max_columns", None)

CANDIDATE_PATHS = ["../../Data/myopia_baseline_clean.csv", "Data/myopia_baseline_clean.csv", "myopia_baseline_clean.csv"]
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("myopia_baseline_clean.csv not found. Checked: " + ", ".join(CANDIDATE_PATHS))
DATA_DIR = os.path.dirname(DATA_PATH) or "."

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "— shape:", df.shape)
assert df.shape == (6392, 14), "Shape does not match Stage 2 output — stop and check your file."


Loaded: ../../Data/myopia_baseline_clean.csv — shape: (6392, 14)


## 2. Per-patient label consistency check

Before splitting by patient, it's important to know whether a patient's two rows (left eye, right eye) always carry the *same* Myopia label. If they don't, that changes how we should think about stratifying the split.

In [2]:
label_nunique_per_patient = df.groupby("ID")["Myopia"].nunique()
n_inconsistent = (label_nunique_per_patient > 1).sum()
print(f"Patients whose two rows have DIFFERENT Myopia labels: {n_inconsistent} / {df['ID'].nunique()}")


Patients whose two rows have DIFFERENT Myopia labels: 45 / 3358


**Finding:** 45 patients have a different `Myopia` value on their left-eye row than on their right-eye row (e.g. left eye myopic, right eye not). Investigating why led to a useful clarification of Stage 2: the `labels` column (used to derive the `Myopia` target) is actually recorded **per image/eye** — it matches whichever specific file that row's `filename` points to — not a single merged label for the whole patient. For example, patient ID 0 has `labels=['C']` on the row for `0_left.jpg` and `labels=['N']` on the row for `0_right.jpg`, matching that eye's own diagnostic keywords exactly.

This does not change any Stage 2 decision or number — the 232-row myopia count is still correct as an image-level count — it just means "the official label" should be understood as *per-image*, not *per-patient*. That's actually appropriate for this project, since each row genuinely is one image.

**Consequence for splitting:** a single patient can legitimately have one myopic eye and one non-myopic eye. So when we stratify the *patient-level* split to preserve class balance, we use a patient-level summary — "does this patient have at least one myopic eye" — purely as a grouping/stratification convenience, not as a claim that both eyes share one label.

In [3]:
patient_table = df.groupby("ID")["Myopia"].max().rename("Patient_Has_Myopia").reset_index()
print("Unique patients:", len(patient_table))
print("Patients with at least one myopic eye:", patient_table["Patient_Has_Myopia"].sum(),
      f"({patient_table['Patient_Has_Myopia'].mean()*100:.2f}%)")


Unique patients: 3358
Patients with at least one myopic eye: 154 (4.59%)


## 3. Patient-level, stratified train/test split (80/20)

In [4]:
RANDOM_STATE = 42

train_ids, test_ids = train_test_split(
    patient_table["ID"],
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=patient_table["Patient_Has_Myopia"],
)

train_ids = set(train_ids)
test_ids = set(test_ids)

print("Patients in train:", len(train_ids))
print("Patients in test:", len(test_ids))

# Sanity check: no patient in both
overlap = train_ids & test_ids
print("Patient ID overlap between train and test:", len(overlap))
assert len(overlap) == 0


Patients in train: 2686
Patients in test: 672
Patient ID overlap between train and test: 0


In [5]:
df["split"] = np.where(df["ID"].isin(train_ids), "train", "test")

print(df["split"].value_counts())
print()
print("Row-level Myopia rate by split:")
print(df.groupby("split")["Myopia"].mean().round(4))
print()
print("Row-level Myopia counts by split:")
print(df.groupby("split")["Myopia"].value_counts())


split
train    5108
test     1284
Name: count, dtype: int64

Row-level Myopia rate by split:
split
test     0.0350
train    0.0366
Name: Myopia, dtype: float64

Row-level Myopia counts by split:
split  Myopia
test   0         1239
       1           45
train  0         4921
       1          187
Name: count, dtype: int64


**Verify no row-level leakage:** every row belonging to a given patient ID must be entirely on one side of the split.

In [6]:
row_level_check = df.groupby("ID")["split"].nunique()
n_split_patients = (row_level_check > 1).sum()
print("Patients whose rows are split across train AND test:", n_split_patients)
assert n_split_patients == 0, "Leakage detected — a patient appears in both splits!"
print("Confirmed: every patient's rows are entirely in one split.")


Patients whose rows are split across train AND test: 0
Confirmed: every patient's rows are entirely in one split.


## 4. Imputation + Standardization for `Patient_Age`, fit on TRAIN only

`Patient_Age` has 28 missing values (from the Stage 2 `Patient Age == 1` placeholder cleanup). The imputer's median and the scaler's mean/std are both computed **only from the training rows**, then applied to test — this is what "fit on training data only" means in practice, and it's what prevents information about the test set from leaking into preprocessing.

In [7]:
train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

print("Missing Patient_Age — train:", train_df["Patient_Age"].isna().sum(),
      "| test:", test_df["Patient_Age"].isna().sum())

train_age_median = train_df["Patient_Age"].median()
print(f"Median age computed from TRAINING data only: {train_age_median}")


Missing Patient_Age — train: 26 | test: 2
Median age computed from TRAINING data only: 59.0


In [8]:
numeric_features = ["Patient_Age"]
categorical_features = ["Patient Sex"]

preprocessor = ColumnTransformer(transformers=[
    ("age", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("sex", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), categorical_features),
])

# Fit ONLY on training rows
preprocessor.fit(train_df[numeric_features + categorical_features])

print("Imputer median learned (should match the value above):",
      preprocessor.named_transformers_["age"].named_steps["impute"].statistics_)
print("Scaler mean / scale learned from training data:",
      preprocessor.named_transformers_["age"].named_steps["scale"].mean_,
      preprocessor.named_transformers_["age"].named_steps["scale"].scale_)
print("Sex categories seen:", preprocessor.named_transformers_["sex"].categories_)


Imputer median learned (should match the value above): [59.]
Scaler mean / scale learned from training data: [58.1468285] [11.1495483]
Sex categories seen: [array(['Female', 'Male'], dtype=object)]


In [9]:
# Transform both sets using the TRAIN-fitted preprocessor
X_train = preprocessor.transform(train_df[numeric_features + categorical_features])
X_test = preprocessor.transform(test_df[numeric_features + categorical_features])

feature_names = ["Patient_Age_Scaled", "Sex_Male"]  # OneHotEncoder(drop="if_binary") keeps 1 column (Male=1)

X_train_df = pd.DataFrame(X_train, columns=feature_names, index=train_df.index)
X_test_df = pd.DataFrame(X_test, columns=feature_names, index=test_df.index)

y_train = train_df["Myopia"]
y_test = test_df["Myopia"]

print("X_train shape:", X_train_df.shape)
print("X_test shape:", X_test_df.shape)
print()
print("Any missing values in X_train?", X_train_df.isna().sum().sum())
print("Any missing values in X_test?", X_test_df.isna().sum().sum())


X_train shape: (5108, 2)
X_test shape: (1284, 2)

Any missing values in X_train? 0
Any missing values in X_test? 0


**Sanity check on the encoding:** confirm `Sex_Male` = 1 corresponds to Male and 0 to Female.

In [10]:
check = train_df[["Patient Sex"]].reset_index(drop=True).join(
    pd.Series(X_train_df["Sex_Male"].values, name="Sex_Male")
)
print(check.drop_duplicates().to_string(index=False))


Patient Sex  Sex_Male
     Female       0.0
       Male       1.0


## 5. Final feature matrix (Experiment 1 model input)

In [11]:
final_train = pd.concat([
    train_df[["ID"]].reset_index(drop=True),
    X_train_df.reset_index(drop=True),
    y_train.reset_index(drop=True),
], axis=1)

final_test = pd.concat([
    test_df[["ID"]].reset_index(drop=True),
    X_test_df.reset_index(drop=True),
    y_test.reset_index(drop=True),
], axis=1)

print("Final Experiment-1 feature matrix — TRAIN:")
print(final_train.shape)
final_train.head()


Final Experiment-1 feature matrix — TRAIN:
(5108, 4)


,ID,Patient_Age_Scaled,Sex_Male,Myopia
0,0,0.973418,0.0,0
1,2,-1.448205,1.0,0
2,5,-0.730687,0.0,0
3,6,0.166210,1.0,0
4,7,0.166210,0.0,0


In [12]:
print("Final Experiment-1 feature matrix — TEST:")
print(final_test.shape)
final_test.head()


Final Experiment-1 feature matrix — TEST:
(1284, 4)


,ID,Patient_Age_Scaled,Sex_Male,Myopia
0,1,-0.102859,1.0,0
1,4,-0.461618,1.0,0
2,11,0.166210,0.0,0
3,16,-0.371928,0.0,1
4,18,-0.013169,1.0,1


In [13]:
print("Train target balance:")
print(y_train.value_counts(), y_train.value_counts(normalize=True).round(4))
print()
print("Test target balance:")
print(y_test.value_counts(), y_test.value_counts(normalize=True).round(4))


Train target balance:
Myopia
0    4921
1     187
Name: count, dtype: int64 Myopia
0    0.9634
1    0.0366
Name: proportion, dtype: float64

Test target balance:
Myopia
0    1239
1      45
Name: count, dtype: int64 Myopia
0    0.965
1    0.035
Name: proportion, dtype: float64


## 6. Export train/test sets

Saved to `Data/train_set.csv` and `Data/test_set.csv`. Each file carries the full row (including reference columns kept for the future multi-class experiment) plus the new preprocessing outputs, so nothing from Stage 2 is lost.

In [14]:
export_cols_extra = ["Age_Was_Missing", "Myopia_OneHot_Reference", "N", "D", "G", "C", "A", "H", "O", "n_conditions"]

train_export = train_df[["ID", "Patient_Age", "Patient Sex"] + export_cols_extra].reset_index(drop=True)
train_export = pd.concat([train_export, X_train_df.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)

test_export = test_df[["ID", "Patient_Age", "Patient Sex"] + export_cols_extra].reset_index(drop=True)
test_export = pd.concat([test_export, X_test_df.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

train_path = os.path.join(DATA_DIR, "train_set.csv")
test_path = os.path.join(DATA_DIR, "test_set.csv")

train_export.to_csv(train_path, index=False)
test_export.to_csv(test_path, index=False)
print("Saved:", train_path, "-", train_export.shape)
print("Saved:", test_path, "-", test_export.shape)


Saved: ../../Data/train_set.csv - (5108, 16)
Saved: ../../Data/test_set.csv - (1284, 16)


In [15]:
# Verify: re-read both files back and re-check the split invariants end to end
check_train = pd.read_csv(train_path)
check_test = pd.read_csv(test_path)

print("Re-read shapes:", check_train.shape, check_test.shape)
assert check_train.shape[0] + check_test.shape[0] == 6392

overlap_check = set(check_train["ID"]) & set(check_test["ID"])
print("ID overlap between saved train/test files:", len(overlap_check))
assert len(overlap_check) == 0

print("Missing values in saved Patient_Age_Scaled — train:", check_train["Patient_Age_Scaled"].isna().sum(),
      "test:", check_test["Patient_Age_Scaled"].isna().sum())

# Confirm Stage 2 file untouched
orig_check = pd.read_csv(DATA_PATH)
assert orig_check.shape == (6392, 14)
print("Verified: myopia_baseline_clean.csv is unchanged —", orig_check.shape)


Re-read shapes: (5108, 16) (1284, 16)
ID overlap between saved train/test files: 0
Missing values in saved Patient_Age_Scaled — train: 0 test: 0
Verified: myopia_baseline_clean.csv is unchanged — (6392, 14)


## 7. Summary

| Item | Value |
|---|---|
| Total rows | 6,392 |
| Total patients | 3,358 |
| Train rows / patients | see output above |
| Test rows / patients | see output above |
| Split method | Patient-level, stratified 80/20 by "patient has any myopic eye" |
| Patient overlap between splits | 0 (verified) |
| Age imputation | Median, computed from training rows only |
| Age scaling | StandardScaler, fit on training rows only |
| Sex encoding | One-hot, binary column `Sex_Male` (1 = Male, 0 = Female) |
| Final Experiment-1 features | `Patient_Age_Scaled`, `Sex_Male` |
| Output files | `Data/train_set.csv`, `Data/test_set.csv` |

**Not started yet:** model training (Stage 4).
